In [16]:
import sys
import gc
import os
import time

import numpy as np
import pandas as pd

from aeon.datasets import load_classification
from aeon.classification.convolution_based import RocketClassifier
from aeon.classification.interval_based import QUANTClassifier
from aeon.classification.distance_based import KNeighborsTimeSeriesClassifier

from sklearn.metrics import accuracy_score

In [17]:
RANDOM_STATE = 42

In [18]:
# List all high dimensional datasets (with more than 1000 dimensions) in the UCR archive.
high_dim_datasets = [
  'ACSF1',
  # 'CinCECGTorso',
  # 'EOGHorizontalSignal',
  # 'EOGVerticalSignal',
  # 'EthanolLevel',
  # 'HandOutlines',
  # 'Haptics',
  # 'HouseTwenty',
  # 'InlineSkate',
  # 'Mallat',
  # 'MixedShapesRegularTrain',
  # 'MixedShapesSmallTrain',
  # 'Phoneme',
  # 'PigAirwayPressure',
  # 'PigArtPressure',
  # 'PigCVP',
  # 'Rock',
  # 'SemgHandGenderCh2',
  # 'SemgHandMovementCh2',
  # 'SemgHandSubjectCh2',
  # 'StarLightCurves',
]

In [19]:
classifiers = {
  "Rocket": RocketClassifier(random_state=RANDOM_STATE),
  # "QUANT": QUANTClassifier(random_state=RANDOM_STATE),
  # "1NN-DTW": KNeighborsTimeSeriesClassifier(n_neighbors=1, distance="dtw"),
  # 'LITE'
}

In [20]:
retention_rates = [
  0.85,
  # 0.70, 
  # 0.55,
  # 0.40,
  # 0.25,
  ]

In [21]:
def PAA_reduce(series, w):
    n = len(series)
    series = np.array(series)
    # Aqui criamos n valores uniformemente espaçados entre 0 e w (exclusivo).
    # Por exemplo: n=6 e w=2 será [0,0,0,1,1,1]
    idx = np.floor(np.linspace(0, w, n, endpoint=False)).astype(int)

    # Aqui iteramos sobre o tamanho desejado 'w' e aplicamos uma máscara para selecionar os pontos da série que pertencem a cada segmento.
    # Por exemplo: n=6 e w=2, o idx resultante será: 
    #              [True, True, True, False, False, False] para w = 0
    #              Assim podemos selecionar os pontos da série que pertencem a cada segmento.
    reduced_series = [np.mean(series[idx == i]) for i in range(w)]

    return reduced_series

In [22]:
reduction_methods = {
  "PAA": PAA_reduce,
}

In [23]:
def znorm(x):
  x_znorm = (x - np.mean(x)) / np.std(x)
  return x_znorm

In [24]:
def load_and_normalize_dataset(dataset_name):
    # Carrega o dataset
    print(f'Carregando dataset {dataset_name}...')
    X_train, y_train = load_classification(dataset_name, split='train')
    X_test, y_test = load_classification(dataset_name, split='test')
    print(f'Dataset {dataset_name} carregado com sucesso. Tamanho do treino: {X_train.shape}, Tamanho do teste: {X_test.shape}')

    # Normaliza os dados usando Z-normalization
    print('Normalizando os dados de treino e teste (Z-normalization)...')
    X_train_normalized = np.array([[znorm(series) for series in sample] for sample in X_train])
    X_test_normalized = np.array([[znorm(series) for series in sample] for sample in X_test])
    print('Normalização concluída!')
    return X_train_normalized, y_train, X_test_normalized, y_test

In [25]:
def train_and_evaluate_classifier(clf_name, clf, X_train, y_train, X_test, y_test):
    # Treina o classificador
    print(f'Treinando o classificador {clf_name}...')
    start = time.time()
    clf.fit(X_train, y_train)
    end = time.time()
    duration = np.round(end - start, 2)
    print(f'Classificador treinado com sucesso em {duration} segundos!')

    # Avalia o classificador
    print(f'Avaliando o classificador {clf_name}...')
    y_pred = clf.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy:.4f}')

    return accuracy, duration

In [26]:
def reduce_dimensionality(method_name, method, X_train, X_test, reduction_rate):
    series_size = X_train.shape[2]
    w = np.round(series_size * (reduction_rate)).astype(int)
    print(f'\nIniciando redução com {method_name} (Novo tamanho: {w})')

    start_time = time.time()

    def run_with_progress(data, label):
        reduced_data = []
        total = len(data)
        for i, sample in enumerate(data):
            reduced_sample = []
            for j, series in enumerate(sample):
                res = method(series, w)
                reduced_sample.append(res)

            reduced_data.append(reduced_sample)
            
            # --- BARRA DE PROGRESSO MANUAL ---
            percent = (i + 1) / total
            bar_length = 30
            done = int(percent * bar_length)
            bar = "█" * done + "-" * (bar_length - done)
            sys.stdout.write(f'\r{label}: [{bar}] {percent:>.1%} ({i+1}/{total})')
            sys.stdout.flush()
            # ----------------------------------
            
        print() # Quebra de linha ao finalizar
        return np.array(reduced_data)

    X_train_reduced = run_with_progress(X_train, "Treino")
    X_test_reduced = run_with_progress(X_test, "Teste ")
    
    end_time = time.time()
    duration = np.round(end_time - start_time, 2)
    print(f'Redução concluída em {duration} segundos!')

    return X_train_reduced, X_test_reduced, duration

In [27]:
########################################################################################################################
#                                                     RODAGEM DE TESTES                                                #
########################################################################################################################
results = []
output_file = 'results_classification.csv'

for dataset in high_dim_datasets:
    try:
        X_train, y_train, X_test, y_test = load_and_normalize_dataset(dataset)
        
        # 1. Dados Originais
        for clf_name, clf in classifiers.items():
            acc, duration = train_and_evaluate_classifier(clf_name, clf, X_train, y_train, X_test, y_test)
            res = {
                'dataset': dataset, 
                'classifier': clf_name, 
                'reduction_method': None,
                'retention_rate': None, 
                'series_size': X_train.shape[2], 
                'accuracy': acc, 
                'training_time': duration,
                'reduction_time': 0
            }
            results.append(res)
            pd.DataFrame([res]).to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)

        # 2. Dados Reduzidos
        for method_name, method in reduction_methods.items():
            for rate in retention_rates:
                X_train_red, X_test_red, reduction_duration = reduce_dimensionality(method_name, method, X_train, X_test, rate)
                
                for clf_name, clf in classifiers.items():
                    acc, duration = train_and_evaluate_classifier(clf_name, clf, X_train_red, y_train, X_test_red, y_test)
                    res = {
                        'dataset': dataset, 
                        'classifier': clf_name, 
                        'reduction_method': method_name,
                        'retention_rate': rate, 
                        'series_size': X_train_red.shape[2], 
                        'accuracy': acc, 
                        'training_time': duration,
                        'reduction_time': reduction_duration
                    }
                    results.append(res)
                    # Salva cada linha imediatamente para segurança
                    pd.DataFrame([res]).to_csv(output_file, mode='a', header=False, index=False)

                # Limpeza pesada após cada taxa de redução/método
                del X_train_red, X_test_red
                gc.collect()

        # Limpeza após processar um dataset inteiro
        del X_train, X_test, y_train, y_test
        gc.collect()
        
    except Exception as e:
        print(f"Erro ao processar dataset {dataset}: {e}")
        continue # Pula para o próximo se um falhar

Carregando dataset ACSF1...
Dataset ACSF1 carregado com sucesso. Tamanho do treino: (100, 1, 1460), Tamanho do teste: (100, 1, 1460)
Normalizando os dados de treino e teste (Z-normalization)...
Normalização concluída!
Treinando o classificador Rocket...
Classificador treinado com sucesso em 10.49 segundos!
Avaliando o classificador Rocket...
Accuracy: 0.8900

Iniciando redução com PAA (Novo tamanho: 1241)
Treino: [██████████████████████████████] 100.0% (100/100)
Teste : [██████████████████████████████] 100.0% (100/100)
Redução concluída em 2.55 segundos!
Treinando o classificador Rocket...
Classificador treinado com sucesso em 8.79 segundos!
Avaliando o classificador Rocket...
Accuracy: 0.8700
